In [1]:
from pathlib import Path
import os
import pandas as pd
import random
#PATHING SCRIPT FOR EVERY EXCERSISE - PROJECT - WORKSPACE
# 1.Literal definition of route pathings(Every user of remote repository must config this pathing in order to find the local repository of his computer)
# My case:
ROOT = Path("/home/josu/Documentos/DataScienceCourse")

# We verify the existence before continue
if not ROOT.exists():
    raise FileNotFoundError(f"❌ The route {ROOT} doesn't exist. Check it.")

# 2. Fix the workspace
os.chdir(ROOT)
print(f"✅ Worskspace enabled!: {os.getcwd()}")

# 3. Define relative pathings to work properly
DATA_TABLES = ROOT / "data" / "raw" / "Tables"

# Let's verify our table folder:
if not DATA_TABLES.exists():
    # If it fails, monitorize the issue: 
    data_dir = ROOT / "data"
    if data_dir.exists():
        print(f"⚠️ the folder 'data' exists but it doesn't have any 'Tables'. 'data' content: {os.listdir(data_dir)}")
    else:
        print(f"⚠️ The folder 'data' doesn't exist on ROOT workspace: {os.listdir(ROOT)}")
    raise FileNotFoundError(f"❌ The tables route {DATA_TABLES} is missing.")

print(f"📂 Tables route detected: {DATA_TABLES}")


✅ Worskspace enabled!: /home/josu/Documentos/DataScienceCourse
📂 Tables route detected: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


In [2]:
# ==========================================================
# UNIVERSAL HTML TABLE SCRAPER
# ----------------------------------------------------------
# Purpose:
#     Download a rendered webpage and extract a table
#     identified by its column headers.
#
# Matching strategy:
#     Header names DO NOT need an exact match.
#     The expected header only needs to be contained
#     inside the real HTML header.
#
# Example:
#
#     HTML Header:
#         "⇅ Official language(s)"
#
#     Expected:
#         "Official language(s)"
#
#     → MATCH
#
# Returns:
#     pandas.DataFrame
# ==========================================================

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd


async def extract_table(
    url: str,
    expected_columns: list[str],
    headless: bool = True,
    executable_path: str = "/usr/bin/chromium",
    wait_until: str = "domcontentloaded"
) -> pd.DataFrame:

    # ------------------------------------------------------
    # Download HTML
    # ------------------------------------------------------

    pw = await async_playwright().start()

    browser = await pw.chromium.launch(
        executable_path=executable_path,
        headless=headless
    )

    page = await browser.new_page()

    await page.goto(
        url,
        wait_until=wait_until
    )

    html = await page.content()

    await browser.close()
    await pw.stop()

    # ------------------------------------------------------
    # Parse HTML
    # ------------------------------------------------------

    soup = BeautifulSoup(html, "lxml")

    tables = soup.find_all("table")

    # ------------------------------------------------------
    # Search matching table
    # ------------------------------------------------------

    for table in tables:

        header = table.find("tr")

        if header is None:
            continue

        cells = header.find_all(["th", "td"])

        html_headers = [
            cell.get_text(" ", strip=True)
            for cell in cells
        ]

        # --------------------------------------------------
        # Check if every expected column exists
        # (partial matching)
        # --------------------------------------------------

        valid = True

        for expected in expected_columns:

            if not any(
                expected.lower() in header.lower()
                for header in html_headers
            ):
                valid = False
                break

        if not valid:
            continue

        # --------------------------------------------------
        # Convert HTML table to DataFrame
        # --------------------------------------------------

        df = pd.read_html(
            StringIO(str(table))
        )[0]

        # --------------------------------------------------
        # Keep only requested columns
        # --------------------------------------------------

        selected_columns = {}

        for expected in expected_columns:

            for real in df.columns:

                if expected.lower() in str(real).lower():

                    selected_columns[expected] = real
                    break

        df = df[list(selected_columns.values())]

        # Rename columns using EXPECTED_COLUMNS names

        df.columns = list(selected_columns.keys())

        return df

    raise ValueError(
        "No matching table was found."
    )

In [3]:
# ==========================================================
# KNOWLEDGE SCRAPER DISPATCHER
# ----------------------------------------------------------
# Purpose:
#     Select the correct extractor according to the
#     source type.
#
# Supported types:
#     - table
#     - hierarchy text (enumerated text lists derivated from a main sustantive)
#
# Returns
# -------
#     pandas.DataFrame
# ==========================================================

async def scrape_source(source):

    source_type = source["type"].lower()

    if source_type == "table":

        return await extract_table(
            url=source["url"],
            expected_columns=source["columns"]
        )

    elif source_type == "hierarchy":

        return await extract_hierarchy(
            url=source["url"],
            root=source["root"]
        )

    else:

        raise ValueError(
            f"Unknown source type: {source_type}"
        )

In [4]:
# ==========================================================
# HIERARCHY EXTRACTOR
# ----------------------------------------------------------
# Purpose:
#     Extract hierarchical text structures from HTML pages
#     and convert them into a relational DataFrame.
#
# Input:
#     url  -> webpage URL
#     root -> hierarchy root title
#
# Output:
#     pandas.DataFrame
#
# Columns:
#     Section
#     Category
#     Item
#     Description
# ==========================================================

from bs4 import BeautifulSoup
import pandas as pd


async def extract_hierarchy(
    url: str,
    root: str
) -> pd.DataFrame:

    # ---------------------------------------------
    # Download HTML
    # ---------------------------------------------

    html = await download_html(url)

    soup = BeautifulSoup(html, "lxml")

    # ---------------------------------------------
    # Locate hierarchy root
    # ---------------------------------------------

    root_heading = None

    for tag in soup.find_all(["h2", "h3"]):

        title = tag.get_text(" ", strip=True)

        if root.lower() in title.lower():

            root_heading = tag
            break

    if root_heading is None:

        raise ValueError(
            f'Root section "{root}" not found.'
        )

    # ---------------------------------------------
    # Walk through hierarchy
    # ---------------------------------------------

    records = []

    current_section = root
    current_category = None

    node = root_heading.find_next()

    while node:

        # Stop when next H2 begins
        if node.name == "h2" and node != root_heading:
            break

        # New category (H3)
        if node.name == "h3":

            current_category = node.get_text(
                " ",
                strip=True
            )

        # Bullet list
        elif node.name == "ul" and current_category is not None:

            for li in node.find_all("li", recursive=False):

                item = li.get_text(
                    " ",
                    strip=True
                )

                records.append({

                    "Section": current_section,

                    "Category": current_category,

                    "Item": item,

                    "Description": ""

                })

        node = node.find_next()

    # ---------------------------------------------
    # DataFrame
    # ---------------------------------------------

    df = pd.DataFrame(records)

    # ---------------------------------------------
    # Cleanup
    # ---------------------------------------------

    del html
    del soup
    del records

    return df

In [5]:
SOURCES = [

    {
        "name": "CountryCodes",

        "type": "table",

        "url": "https://es.wikipedia.org/wiki/ISO_3166-1_alfa-2",

        "columns": [

    "Código",
    "Nombre del país",
    "Año",
    "ccTLD",
    "ISO 3166-2",
    "Notas"

],

        "output": "CountryCodes.csv"
    },

    {
        "name": "OfficialLanguages",

        "type": "table",

        "url": "https://en.wikipedia.org/wiki/List_of_official_languages_by_country_and_territory",

        "columns": [

    "Country/Region",
    "Number of official (including de facto)",
    "Official language(s)",
    "National language(s)",
    "Regional language(s)",
    "Minority language(s)",
    "Widely spoken"

],

        "output": "OfficialLanguages.csv"
    },

    {
        "name": "EuropeRegions",

        "type": "hierarchy",

        "url": "https://en.wikipedia.org/wiki/Regions_of_Europe",

        "root": "Geographical",

        "output": "EuropeRegions.csv"
    }

]

In [6]:
datasets = {}

for source in SOURCES:

    print(f"Processing: {source['name']}")

    datasets[source["name"]] = await scrape_source(source)

    print(datasets[source["name"]].head())

Processing: CountryCodes
  Código               Nombre del país   Año ccTLD     ISO 3166-2  \
0     AD                       Andorra  1974   .ad  ISO 3166-2:AD   
1     AE  Emiratos Árabes Unidos (los)  1974   .ae  ISO 3166-2:AE   
2     AF                    Afganistán  1974   .af  ISO 3166-2:AF   
3     AG             Antigua y Barbuda  1974   .ag  ISO 3166-2:AG   
4     AI                       Anguila  1985   .ai  ISO 3166-2:AI   

                                               Notas  
0                                                NaN  
1                                                NaN  
2                                                NaN  
3                                                NaN  
4  AI antes representaba al Territorio Francés de...  
Processing: OfficialLanguages
         Country/Region  Number of official (including de facto)  \
0           Abkhazia[a]                                        2   
1  Afghanistan[1][2][3]                                        2

NameError: name 'download_html' is not defined